---
title: "Datavoorbewerking van responsdata"
title-block-style: plain
author: "Jalisa Shaline van der Zeeuw"
date: "January 2026"
format:
  pdf:
    geometry: margin=2.5cm
    include-code: false
number-sections: false
execute:
  echo: false
  warning: false
  message: false
---

In [2]:
import sys

EXPECTED_ENV = "ml_oncoresponse"

if EXPECTED_ENV not in sys.prefix:
    print(
        "⚠️ WAARSCHUWING: mogelijk verkeerde conda environment\n"
        f"Verwacht: {EXPECTED_ENV}\n"
        f"Huidig: {sys.prefix}\n"
        "Resultaten kunnen afwijken."
    )

In dit notebook wordt de responsdata voorbereid voor integratie met de mutatie- en methylatiematrices. 

Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv - bevat therapieresponsgegevens (AUC-waarden) die aangeven hoe gevoelig een cellijn is voor een bepaald medicijn.

[extra uitleg over AUC, nog visueel maken]

Preprocessing bestaat uit:
- selectie van relevantie cellijnen (longkanker)
- selectie van relevante kolommen (`ModelID + AUC-waarden)
- harmonisatie van identifiers
- kwaliteitscontrole
- en het opslaan van de uiteindelijke responsmatrix

#### Stap 1. Inlezen en verkennen van de responswaarden-data


Doel: Het inlezen van de ruwe responsdata, en een eerste verkenning van de vorm en beschikbare kolommen. 

In [12]:
import pandas as pd

# aanmaken variabele met de bestandpad
response_file = "../data/raw/Drug_sensitivity_AUC_(PRISM_Repurposing_Secondary_Screen)_subsetted.csv"
# inlezen
response_df = pd.read_csv(response_file, sep=',') # comma seperated

# verkenning
print("Aantal rijen en kolommen:", response_df.shape) # geeft het aantal rijen en kolommen

# eerste en laatste 10 kolommen tonen als voorbeeld

print(response_df.iloc[:8, :10])


Aantal rijen en kolommen: (480, 1457)
    depmap_id cell_line_display_name              lineage_1  \
0  ACH-000973                   639V  Bladder/Urinary Tract   
1  ACH-001016                 BECKER              CNS/Brain   
2  ACH-000927                  BT474                 Breast   
3  ACH-000288                  BT549                 Breast   
4  ACH-000212                 CAL120                 Breast   
5  ACH-000856                  CAL51                 Breast   
6  ACH-000783                  CAMA1                 Breast   
7  ACH-000117                EFM192A                 Breast   

                   lineage_2                          lineage_3  \
0            Urethral Cancer      Urethral Urothelial Carcinoma   
1             Diffuse Glioma                        Astrocytoma   
2  Invasive Breast Carcinoma   Breast Invasive Ductal Carcinoma   
3  Invasive Breast Carcinoma     Breast Invasive Carcinoma, NOS   
4  Invasive Breast Carcinoma          Invasive Breast Carci

De response-dataset bevat 480 rijen waarbij elke rij een unieke cellijn representeert, en 1457 kolommen. De eerste kolommen bevatten meta-data over de cellijnen:

- depmap_id: unieke ID voor elke cellijn (bijvoorbeeld ACH-000973)  
- cell_line_display_name: leesbare naam van de cellijn (bijvoorbeeld 639V of BECKER)  
- lineage_1 t/m lineage_6: beschrijving van de biologische afkomst van de cellijn, van breed tot specifiek.  

Vanaf kolom 8 bevatten de overige kolommen de drug-response waarden voor verschillende geneesmiddelen, weergegeven als numerieke scores (AUC). Elke cellijn is getest voor meerdere geneesmiddelen, waarbij sommige waarden ontbreken (NaN).

#### Stap 2. Filteren op relevante data

Doel:  Alleen longkankercellijnen behouden en relevante kolommen selecteren. Alle andere metadata kolommen worden verwijderd om een compacte matrix te creëren.

Aanpak:
Dit wordt gedaan door eerst te filteren op de kolom `lineage_1 == "Lung"`. Vervolgens selecteren van de kolom `depmap_id` (later hernoemd naar ModelID) en alle kolommen met geneesmiddel-responswaarden (AUC). De overige metadata kolommen worden verwijderd. (`cell_line_display_name`, `lineage_2 t/m lineage_6`)

In [27]:
# kolommen selecteren (identifiers en AUC-waarden)
id_col = "depmap_id"  # wordt later hernoemd naar ModelID
exclude_cols = ["cell_line_display_name", "lineage_1", "lineage_2", 
                "lineage_3", "lineage_4", "lineage_5", "lineage_6"]
auc_cols = [col for col in response_df.columns if col not in [id_col] + exclude_cols]

# filter longkankercellijnen en selecteer relevante kolommen
lung_response_df = response_df.loc[
    response_df["lineage_1"].str.lower() == "lung",
    [id_col] + auc_cols
].copy()

# controle
print(f"Aantal longkankercellijnen: {lung_response_df.shape[0]}")
print(f"Aantal geneesmiddelen (AUC-kolommen): {len(auc_cols)}")
display(lung_response_df.head())


Aantal longkankercellijnen: 93
Aantal geneesmiddelen (AUC-kolommen): 1450


,depmap_id,8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6),NORETYNODREL (BRD:BRD-A00758722-001-04-9),PREDNISOLONE-ACETATE (BRD:BRD-A01643550-001-04-9),BETAMETHASONE (BRD:BRD-A02180903-001-04-5),MEPIVACAINE (BRD:BRD-A03216249-003-24-3),XL888 (BRD:BRD-A03506276-001-01-5),METOPROLOL (BRD:BRD-A03623303-045-09-5),METHSCOPOLAMINE (BRD:BRD-A03932035-004-04-3),LAPPACONITE (BRD:BRD-A05906449-004-01-1),...,LORLATINIB (BRD:BRD-K99879819-001-02-1),HEXYLRESORCINOL (BRD:BRD-K99946902-001-07-5),BOSULIF (BRD:BRD-K99964838-001-11-9),AMMONIUM-LACTATE (BRD:BRD-M29182745-001-01-4),NEMONAPRIDE (BRD:BRD-M80207679-001-01-5),CROMAKALIM (BRD:BRD-M89827113-001-01-5),EFONIDIPINE-MONOETHANOLATE (BRD:BRD-M92675308-003-07-1),DICHLOROACETATE (BRD:BRD-M97302542-001-04-4),TYLOXAPOL (BRD:BRD-U25960968-000-01-9),SEVELAMER (BRD:BRD-U45393375-000-01-6)
15,ACH-000840,NaN,0.986123,NaN,NaN,0.924684,0.585858,NaN,0.937877,0.866803,...,0.958516,0.907903,0.880807,NaN,NaN,0.984182,0.921305,NaN,NaN,NaN
30,ACH-000921,NaN,0.913647,NaN,NaN,NaN,0.565380,NaN,0.940083,0.882829,...,0.963600,NaN,0.903282,NaN,NaN,0.897773,0.942853,NaN,NaN,NaN
89,ACH-000454,NaN,0.934788,NaN,0.978968,NaN,0.662990,NaN,NaN,0.838097,...,NaN,0.669239,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,ACH-000868,NaN,0.790187,NaN,0.690992,NaN,0.552618,NaN,NaN,0.781147,...,0.840573,NaN,0.912818,0.857715,NaN,0.948024,0.789514,0.925365,NaN,NaN
93,ACH-000901,NaN,0.921198,NaN,NaN,NaN,0.558377,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.953131,NaN,NaN


#### Stap 3. Controle op missing values


Doel: Zorgen dat de responsmatrix volledig bruikbaar is voor modeltraining.

Omdat het hier om de outputbariabele (AUC) gaat, mogen er geen missende waarden zijn. Elke missende waarde betekent dus dat die hele rij (cellijn) of kolom (geneesmiddel) niet kunt gebruiken. XXXXXX?

We beginnen met het berekenen van het aantal ontbrekende waarden (NaN) per geneesmiddel (kolom). Zo krijgen we inziht in welke geneesmiddelen volledig ontbreken voor de longkankercellijnen en welke gedeeltelijk.

In [28]:
# aantal missende waarden per kolom
missing_counts = lung_response_df.isna().sum().sort_values(ascending=False)

print("Aantal missing values per kolom (top 10):")
print(missing_counts.head(10))  # meest missende waarden
print("\nAantal missing values per kolom (minst 10):")
print(missing_counts.tail(10))  # minst missende waarden


Aantal missing values per kolom (top 10):
ABIRATERONE (BRD:BRD-K50071428-001-03-3)                         93
BAN-ORL-24 (BRD:BRD-K47049295-300-01-5)                          93
SODIUM-TANSHINONE-II-A-SULFONATE (BRD:BRD-K84798689-236-02-9)    93
SULFAMETHAZINE (BRD:BRD-K11640013-236-10-1)                      93
NELARABINE (BRD:BRD-K84466663-001-05-4)                          93
JNJ-16259685 (BRD:BRD-K25596805-001-01-8)                        93
METOPROLOL (BRD:BRD-A03623303-045-09-5)                          93
KY02111 (BRD:BRD-K13302470-001-02-7)                             93
NIRIDAZOLE (BRD:BRD-K53123955-001-10-5)                          93
RG108 (BRD:BRD-K89391146-001-08-0)                               93
dtype: int64

Aantal missing values per kolom (minst 10):
KW-2478 (BRD:BRD-K41213548-001-02-0)        1
GDC-0349 (BRD:BRD-K43187018-001-03-3)       1
RUBITECAN (BRD:BRD-K79821389-001-03-5)      0
SB-2343 (BRD:BRD-K98795921-001-01-7)        0
OTS167 (BRD:BRD-K53417444-003-03-1) 

Observaties en keuze van geneesmiddel

Observaties:
- Slechts enkele geneesmiddelen hebben geen missende waarden voor ale 93 longkankercellijnen.
- De meeste andere geneesmiddelen missen waarden in een deel of alle cellijnen.
- Voor de meeste geneesmiddelen zou ik dus een deel van de cellijnen moeten verwijderen, waardoor het aantal samples kleiner wordt.

Omdat het aantal longkankercellijnen al heel beperkt is (n = 93), is er gekozen voor een geneesmiddel zonder missende responswaarden. Hierdoor blijven alle beschikbare samples behouden voor het trainen en evalueren van het Random Forest-model. 

Naast datakwaliteit en samplegrootte is ook gekeke naar het biologische werkingsmechanisme van de oovergebleven geneesmiddelen. In een latere fase van het project wordt pathway-aggregatie toegepast om de grote hoeveelheid mutatie- en methylatiefeatuers te reduceren en biologisch interpreteerbaar te maken. Daarom is het wenselijk om een geneesmiddel te kiezen dat aangrijpt op een duidelijk gedefinieerde en goed beschreven biologische pathway. 

Twee geneesmiddelen met volledige data voldoen aan deze criteria:
- PF-05212384, een duale PI3K/mTOR-remmer, die aangrijpt op een van de meest centrale oncogenese signaalroutes.
- Verzosertib, een ATR-kinaseremmer, die betrokken is bij DNA-schadeherstel en celcycluscontrole.

Dezed pathways zijn goed beschreven en bevatten meerdere genen, waardoor zij geschikt zijn voor pathway-aggregatie. Dit maakt het mogelijk om te onderzoeken of veranderingen in mutatie- en methylatiepatronen binnen relevante pathways samenhangen met geneesmiddelgevoeligheid. Op deze manier fungeert de keuze van het geneesmiddel ook als een soort controle van het model.

In dit project is gekozen om te starten met PF-05212384, omdat de PI3K/mTOR-pathway uitgebreid is bestudeerd in kanker en vaak verstoort is in longkanker. Dit zal het makkelijker maken om de resultaten van het model eenvoudiger te interpreteren en te relateren aan de bestaande biologische kennis. Bezosertib wordt beschouwd als een mogelijke vervolganalyse om te vergelijken of het Random Forest-model voor een geneesmiddel met een ander werkingsmechanisme andere biologische pathways als belangrijk identificeert. 

Deze afwegingen laten zien dat de geneesmiddelkeuze gebaseerd is op een combinatie van datacompleetheid, behoud van samplegrootte en biologische interpretatie. 








#### Stap 4. Harmoniseren van identifiers 


Doel: De responsdataset voorbereiden op integratie met de mutatie- en methylatiedata door het gebruik van de gedeelde identifier (`ModelID`). Dit zorgt ervoor dat alle datasets zzonder problemen kunnen worden samengevoegd op cellijnniveau.

In de oorspronkelijke responsdataset worden cellijnen geïdentificeerd via de kolom `depmap_id`. In de mutatie- en methylatiedata wordt de naam `ModelID` (DepMap/ACH-ID) gebruikt als standaardidentifier. 

Aanpak:
- hernoemen van `depmap_id` naar `ModelID`.
- controleren van de structuur van de dataset na harmoniseren.

In [31]:
# hernoem depmap_id naar ModelID voor consistente identifiers
lung_response_df = lung_response_df.rename(columns={"depmap_id": "ModelID"})

# controle van de kolommen
print("Eerste kolommen na harmonisatie:")
print(lung_response_df.columns[:5])

# controle van het aantal cellijnen
print(f"\nAantal cellijnen in responsmatrix: {lung_response_df.shape[0]}")
print(f"Aantal responskolommen (geneesmiddelen): {lung_response_df.shape[1] - 1}")

# opschonen kolomnamen (verwijder alles vanaf de eerste haak
lung_response_df.columns = [
    col.split(" (")[0] for col in lung_response_df.columns
]

# voorbeeld van de dataset
display(lung_response_df.head())

Eerste kolommen na harmonisatie:
Index(['ModelID', '8-BROMO-CGMP (BRD:BRD-A00077618-236-07-6)',
       'NORETYNODREL (BRD:BRD-A00758722-001-04-9)',
       'PREDNISOLONE-ACETATE (BRD:BRD-A01643550-001-04-9)',
       'BETAMETHASONE (BRD:BRD-A02180903-001-04-5)'],
      dtype='object')

Aantal cellijnen in responsmatrix: 93
Aantal responskolommen (geneesmiddelen): 1450


,ModelID,8-BROMO-CGMP,NORETYNODREL,PREDNISOLONE-ACETATE,BETAMETHASONE,MEPIVACAINE,XL888,METOPROLOL,METHSCOPOLAMINE,LAPPACONITE,...,LORLATINIB,HEXYLRESORCINOL,BOSULIF,AMMONIUM-LACTATE,NEMONAPRIDE,CROMAKALIM,EFONIDIPINE-MONOETHANOLATE,DICHLOROACETATE,TYLOXAPOL,SEVELAMER
15,ACH-000840,NaN,0.986123,NaN,NaN,0.924684,0.585858,NaN,0.937877,0.866803,...,0.958516,0.907903,0.880807,NaN,NaN,0.984182,0.921305,NaN,NaN,NaN
30,ACH-000921,NaN,0.913647,NaN,NaN,NaN,0.565380,NaN,0.940083,0.882829,...,0.963600,NaN,0.903282,NaN,NaN,0.897773,0.942853,NaN,NaN,NaN
89,ACH-000454,NaN,0.934788,NaN,0.978968,NaN,0.662990,NaN,NaN,0.838097,...,NaN,0.669239,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,ACH-000868,NaN,0.790187,NaN,0.690992,NaN,0.552618,NaN,NaN,0.781147,...,0.840573,NaN,0.912818,0.857715,NaN,0.948024,0.789514,0.925365,NaN,NaN
93,ACH-000901,NaN,0.921198,NaN,NaN,NaN,0.558377,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.953131,NaN,NaN


De identifiers zijn nu consistent (`ModelID`).

Na harmonisatie gebruikt de responsdataset dezelfde identifier (`ModelID`) als de mutatie- en methylatiedatasets. Hierdoor is de dataset technisch gereed voor integratie. De responsmatrix bevat 93 longkankercellijnen (rijen) en 1450 geneesmiddelen (kolommen met AUC-waarden). 

Er is bewust nog geen selectie gemaakt voor een specifiek geneesmiddel. Dit zorgt voor een flexibele en reproduceerbare workflow, waarbij dezelfde responsdataset kan worden hergebruikt voor verschillende modellen of analysemogelijkheden. 

In [33]:
import os

# zet ModelID als index voordat we opslaan
lung_response_df = lung_response_df.set_index("ModelID")

# output map voor verwerkte data (hier komt de pickle terecht)
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# sla de geharmoniseerde responsmatrix op als pickle
lung_response_df.to_pickle(
    os.path.join(output_dir, "response_matrix.pkl"))
